In [1]:
!pip install torch transformers tree_sitter==0.21.3 scikit-learn tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 9.2 MB/s eta 0:00:00


In [2]:
import os
import json
import torch
import logging
import random
import numpy as np
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset, RandomSampler, SequentialSampler, random_split
from torch.optim import AdamW
from transformers import (get_linear_schedule_with_warmup,
                          RobertaConfig, RobertaModel, AutoTokenizer)
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)

# ─── CONFIGURATION ────────────────────────────────────────────────────────
class Args:
    output_dir         = "saved_models_unixcoder_dfg"
    model_name_or_path = "microsoft/unixcoder-base"

    # Data path
    train_file         = "/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl"

    # Sequence lengths (same as GraphCodeBERT for fair comparison)
    code_length        = 256
    data_flow_length   = 64

    # Hyperparameters
    train_batch_size   = 8
    eval_batch_size    = 16
    learning_rate      = 2e-5
    max_grad_norm      = 1.0
    num_train_epochs   = 5
    seed               = 42

    num_workers = 0
    patience = 2
    best_model_path = os.path.join(output_dir, "model_unixcoder_dfg_best.bin")
    results_path = os.path.join(output_dir, "unixcoder_dfg_results.txt")
    split_indices_path = os.path.join(output_dir, "unixcoder_dfg_split_indices.json")
    split_summary_path = os.path.join(output_dir, "unixcoder_dfg_split_summary.json")
    history_path = os.path.join(output_dir, "unixcoder_dfg_training_history.json")

    # In Args:
    gradient_accumulation_steps = 2  # effective batch = 8 × 2 = 16

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

args = Args()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(args.seed)
print(f"Device : {args.device}")
print(f"Model  : {args.model_name_or_path}")
print(f"Input  : code_length={args.code_length}  dfg_length={args.data_flow_length}")


Device : cuda
Model  : microsoft/unixcoder-base
Input  : code_length=256  dfg_length=64


In [3]:
# ─── DFG-AWARE MODEL ─────────────────────────────────────────────────────
# UniXcoder uses the same RoBERTa-style internal API as GraphCodeBERT,
# so the custom sparse attention forward pass works without modification.
class Model(nn.Module):
    def __init__(self, encoder, config):
        super(Model, self).__init__()
        self.encoder    = encoder
        self.config     = config
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids=None, p_ids=None, attn_mask=None, labels=None):
        # Convert boolean mask: 1 = attend, 0 = mask -> 0 / -10000
        extended_attention_mask = (1.0 - attn_mask) * -10000.0
        extended_attention_mask = extended_attention_mask.unsqueeze(1)
        # Shape: [Batch, 1, Seq, Seq]

        # Embed tokens with custom position IDs (DFG nodes use pos=0)
        embedding_output = self.encoder.embeddings(
            input_ids=input_ids,
            position_ids=p_ids
        )

        # Pass through transformer layers with the sparse mask
        encoder_outputs = self.encoder.encoder(
            embedding_output,
            attention_mask=extended_attention_mask,
            head_mask=[None] * self.config.num_hidden_layers
        )

        sequence_output = encoder_outputs[0]
        logits = self.classifier(self.dropout(sequence_output[:, 0, :]))
        prob   = F.softmax(logits, dim=-1)

        if labels is not None:
            loss = CrossEntropyLoss()(logits, labels)
            return loss, prob
        return prob


In [4]:
# ─── DFG-AWARE DATASET ───────────────────────────────────────────────────
class TextDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.args      = args
        self.tokenizer = tokenizer
        self.total_len = args.code_length + args.data_flow_length

        with open(file_path, 'r') as f:
            self.lines = f.readlines()

    def __len__(self):
        return len(self.lines)

    def _get_char_index(self, code_lines, coord):
        row, col = coord
        char_idx = 0
        for i in range(min(row, len(code_lines))):
            char_idx += len(code_lines[i])
        return char_idx + col

    def __getitem__(self, item):
        entry      = json.loads(self.lines[item])
        code       = entry.get('code', '')
        dfg        = entry.get('dfg', [])[:self.args.data_flow_length]
        label      = int(entry.get('label', 0)) if entry.get('label') is not None else 0
        code_lines = code.splitlines(keepends=True)

        # ── 1. Tokenize with offset mapping for DFG alignment ─────────────
        tokens_obj = self.tokenizer(
            code,
            max_length=self.args.code_length,
            truncation=True,
            padding='max_length',
            return_offsets_mapping=True
        )
        input_ids = tokens_obj['input_ids']
        offsets   = tokens_obj['offset_mapping']

        # ── 2. Map DFG nodes to code tokens ───────────────────────────────
        dfg_ids          = [self.tokenizer.unk_token_id] * len(dfg)
        node_to_token_map = {}
        pos_to_node_idx   = {}

        for node_idx, node_item in enumerate(dfg):
            try:
                start_pos = node_item[1][0]
                end_pos   = node_item[1][1]
                if len(start_pos) < 2 or len(end_pos) < 2:
                    node_to_token_map[node_idx] = []
                    continue
                pos_key = (start_pos[0], start_pos[1], end_pos[0], end_pos[1])
                pos_to_node_idx[pos_key] = node_idx

                char_start = self._get_char_index(code_lines, start_pos)
                char_end   = self._get_char_index(code_lines, end_pos)

                aligned = []
                for t_idx, (t_start, t_end) in enumerate(offsets):
                    if t_start == t_end:
                        continue
                    if (t_start >= char_start and t_end <= char_end) or \
                       (char_start >= t_start and char_end <= t_end):
                        aligned.append(t_idx)
                node_to_token_map[node_idx] = aligned
            except (IndexError, TypeError):
                node_to_token_map[node_idx] = []

        # ── 3. Build sparse attention mask ────────────────────────────────
        c_len    = self.args.code_length
        attn_mask = np.zeros((self.total_len, self.total_len), dtype=bool)

        # A. Code tokens attend to each other freely
        attn_mask[:c_len, :c_len] = True

        # B. DFG node <-> aligned code tokens (bidirectional anchor)
        # C. DFG node <-> parent nodes along data-flow edges (bidirectional)
        # D. Self-loop for every DFG node
        for node_idx, node_item in enumerate(dfg):
            abs_node = c_len + node_idx

            for t_idx in node_to_token_map.get(node_idx, []):
                attn_mask[abs_node, t_idx] = True
                attn_mask[t_idx, abs_node] = True

            try:
                for p_pos in node_item[4]:
                    if len(p_pos) < 2:
                        continue
                    p_key = (p_pos[0][0], p_pos[0][1], p_pos[1][0], p_pos[1][1])
                    if p_key in pos_to_node_idx:
                        abs_parent = c_len + pos_to_node_idx[p_key]
                        attn_mask[abs_node, abs_parent] = True
                        attn_mask[abs_parent, abs_node] = True
            except (IndexError, TypeError):
                pass

            attn_mask[abs_node, abs_node] = True

        # ── 4. Assemble final sequence ─────────────────────────────────────
        full_input_ids = input_ids + dfg_ids
        # Code tokens: sequential positions starting at 2 (0=pad, 1=special)
        # DFG nodes:   fixed position 0 (non-sequential signal)
        p_ids = [i + 2 for i in range(c_len)] + [0] * len(dfg_ids)

        pad = self.total_len - len(full_input_ids)
        if pad > 0:
            full_input_ids += [self.tokenizer.pad_token_id] * pad
            p_ids          += [1] * pad

        return {
            'input_ids': torch.tensor(full_input_ids, dtype=torch.long),
            'p_ids':     torch.tensor(p_ids,          dtype=torch.long),
            'attn_mask': torch.tensor(attn_mask,      dtype=torch.float),
            'label':     torch.tensor(label,          dtype=torch.long)
        }


In [5]:
def load_entries(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        raw = f.read().strip()
    if not raw:
        return []
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, list):
            return parsed
    except json.JSONDecodeError:
        pass
    entries = []
    for line in raw.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            entries.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    if not entries:
        raise ValueError(f'Could not parse dataset file as JSON array or JSONL: {filepath}')
    return entries


In [6]:
def evaluate(model, dataset, args, tag="Eval", quiet=False):
    dataloader = DataLoader(dataset, sampler=SequentialSampler(dataset), batch_size=args.eval_batch_size, num_workers=args.num_workers, pin_memory=True)
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Evaluating {tag}", disable=quiet):
            probs = model(input_ids=batch['input_ids'].to(args.device), p_ids=batch['p_ids'].to(args.device), attn_mask=batch['attn_mask'].to(args.device))
            all_probs.append(probs.cpu().numpy())
            all_labels.extend(batch['label'].cpu().numpy())
    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.array(all_labels)
    all_preds = np.argmax(all_probs, axis=-1)
    acc = accuracy_score(all_labels, all_preds)
    roc_auc = roc_auc_score(all_labels, all_probs[:, 1])
    pr_auc = average_precision_score(all_labels, all_probs[:, 1])
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    if not quiet:
        print("\n" + "=" * 40)
        print(f"RESULTS ({tag})")
        print("=" * 40)
        print(f"Accuracy : {acc:.4%}")
        print(f"ROC-AUC : {roc_auc:.4f}")
        print(f"PR-AUC : {pr_auc:.4f}")
        print(f"FN Count : {fn}")
        print(f"FP Count : {fp}")
        print(classification_report(all_labels, all_preds, target_names=['Safe', 'Vuln'], digits=4))
        print("Confusion Matrix:")
        print(confusion_matrix(all_labels, all_preds))
    return {'probs': all_probs, 'labels': all_labels, 'acc': acc, 'roc_auc': roc_auc, 'pr_auc': pr_auc, 'fn': int(fn), 'fp': int(fp)}

def train(model, train_dataset, val_dataset, args):
    train_dataloader = DataLoader(
        train_dataset,
        sampler=RandomSampler(train_dataset),
        batch_size=args.train_batch_size,
        num_workers=args.num_workers,
        pin_memory=True,
        drop_last=True
    )
    optimizer = AdamW(model.parameters(), lr=args.learning_rate, eps=1e-8)
    total_steps = (len(train_dataloader) // args.gradient_accumulation_steps) * args.num_train_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.1),
        num_training_steps=total_steps
    )
    scaler = GradScaler(enabled=torch.cuda.is_available())
    best_val_acc = -1.0
    best_epoch = 0
    patience_counter = 0
    history = []

    for epoch in range(args.num_train_epochs):
        model.train()
        tr_loss = 0.0
        optimizer.zero_grad(set_to_none=True)  # moved outside step loop
        bar = tqdm(train_dataloader, desc=f"Epoch {epoch + 1}")

        for step, batch in enumerate(bar):
            with autocast():
                loss, _ = model(
                    input_ids=batch['input_ids'].to(args.device),
                    p_ids=batch['p_ids'].to(args.device),
                    attn_mask=batch['attn_mask'].to(args.device),
                    labels=batch['label'].to(args.device)
                )
                loss = loss / args.gradient_accumulation_steps  # normalize loss

            scaler.scale(loss).backward()
            tr_loss += loss.item() * args.gradient_accumulation_steps  # re-scale for logging

            if (step + 1) % args.gradient_accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            bar.set_postfix(loss=tr_loss / (step + 1))

        avg_train_loss = tr_loss / len(train_dataloader)
        val_metrics = evaluate(model, val_dataset, args, tag=f"Validation Epoch {epoch + 1}")
        val_acc = val_metrics['acc']
        history.append({
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'val_acc': val_acc,
            'val_roc_auc': val_metrics['roc_auc'],
            'val_pr_auc': val_metrics['pr_auc'],
            'val_fn': val_metrics['fn'],
            'val_fp': val_metrics['fp']
        })

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save(model.state_dict(), args.best_model_path)
        else:
            patience_counter += 1
            if patience_counter >= args.patience:
                break

    return best_epoch, best_val_acc, history

In [7]:
from collections import defaultdict, Counter
import math, random, json, os

# Build dataset and stratified 3:1:3:1 split

tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path, use_fast=True)
full_dataset = TextDataset(tokenizer, args, args.train_file)

def load_entries(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        raw = f.read().strip()
    if not raw:
        return []
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, list):
            return parsed
    except json.JSONDecodeError:
        pass
    entries = []
    for line in raw.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            entries.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return entries

entries = load_entries(args.train_file)
assert len(entries) == len(full_dataset), f"Dataset size mismatch: {len(entries)} vs {len(full_dataset)}"

def infer_source(entry):
    for key in ("source", "dataset", "origin", "project"):
        value = entry.get(key)
        if value is not None and str(value).strip() != "":
            return str(value).strip()
    return "unknown"

def allocate_counts(total_needed, groups, fraction):
    raw = {g: len(v) * fraction for g, v in groups.items()}
    base = {g: int(math.floor(v)) for g, v in raw.items()}
    remainder = total_needed - sum(base.values())
    order = sorted(groups.keys(), key=lambda g: (raw[g] - base[g], len(groups[g])), reverse=True)
    for g in order[:remainder]:
        base[g] += 1
    return base

def stratified_three_way_split(entries, test_ratio=0.10, val_ratio=0.08, seed=42):
    rng = random.Random(seed)
    source_to_indices = defaultdict(list)
    for idx, entry in enumerate(entries):
        source_to_indices[infer_source(entry)].append(idx)
    for indices in source_to_indices.values():
        rng.shuffle(indices)

    total = len(entries)
    target_test = int(round(total * test_ratio))
    target_val = int(round(total * val_ratio))
    target_train = total - target_test - target_val

    test_alloc = allocate_counts(target_test, source_to_indices, test_ratio)
    test_indices = []
    trainval_groups = {}
    for source, indices in source_to_indices.items():
        take = min(test_alloc[source], len(indices))
        test_indices.extend(indices[:take])
        trainval_groups[source] = indices[take:]

    adjusted_val_ratio = val_ratio / (1.0 - test_ratio)
    val_alloc = allocate_counts(target_val, trainval_groups, adjusted_val_ratio)

    val_indices, train_indices = [], []
    for source, indices in trainval_groups.items():
        take = min(val_alloc[source], len(indices))
        val_indices.extend(indices[:take])
        train_indices.extend(indices[take:])

    train_indices = sorted(train_indices)
    val_indices = sorted(val_indices)
    test_indices = sorted(test_indices)
    assert len(train_indices) == target_train
    assert len(val_indices) == target_val
    assert len(test_indices) == target_test
    return train_indices, val_indices, test_indices

train_indices, val_indices, test_indices = stratified_three_way_split(
    entries, test_ratio=0.10, val_ratio=0.08, seed=args.seed
)

train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
val_dataset = torch.utils.data.Subset(full_dataset, val_indices)
test_dataset = torch.utils.data.Subset(full_dataset, test_indices)

print("Dataset split ready")
print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))
print("Train sources:", dict(Counter(infer_source(entries[i]) for i in train_indices)))
print("Val sources:", dict(Counter(infer_source(entries[i]) for i in val_indices)))
print("Test sources:", dict(Counter(infer_source(entries[i]) for i in test_indices)))

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/vocab.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/vocab.json "HTTP/1.1 200 OK"


...

merges.txt: 0.00B [00:00, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/tokenizer.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/special_tokens_map.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


Dataset split ready
Train: 163967
Val: 15997
Test: 19996
Train sources: {'unknown': 163967}
Val sources: {'unknown': 15997}
Test sources: {'unknown': 19996}


In [8]:
config = RobertaConfig.from_pretrained(args.model_name_or_path)
config.num_labels = 2
encoder = RobertaModel.from_pretrained(args.model_name_or_path, config=config)
model = Model(encoder, config)
model.to(args.device)

# Creates the output folder
os.makedirs(args.output_dir, exist_ok=True)

best_epoch, best_val_acc, history = train(model, train_dataset, val_dataset, args)

model.load_state_dict(torch.load(args.best_model_path, map_location=args.device))
final_metrics = evaluate(model, test_dataset, args, tag="Final Test", quiet=False)

np.save('/kaggle/working/unixcoder_dfg_test_probs.npy', final_metrics['probs'])
np.save('/kaggle/working/unixcoder_dfg_test_labels.npy', final_metrics['labels'])

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/model.safetensors

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/commits/main "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/discussions?p=0 "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/commits/refs%2Fpr%2F8 "HTTP/1.1 200 OK"
RobertaModel LOAD REPORT from: microsoft/unixcoder-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/refs%2Fpr%2F8/model.safetensors.index.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/refs%2Fpr%2F8/model.safetensors "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/xet-read-token/558aa506226b13a7e83f66cb38c53e61b9603eac "HTTP/1.1 200 OK"


...


RESULTS (Validation Epoch 4)
Accuracy : 88.7104%
ROC-AUC : 0.9594
PR-AUC : 0.9600
FN Count : 878
FP Count : 928
              precision    recall  f1-score   support

        Safe     0.8892    0.8836    0.8864      7972
        Vuln     0.8851    0.8906    0.8878      8025

    accuracy                         0.8871     15997
   macro avg     0.8871    0.8871    0.8871     15997
weighted avg     0.8871    0.8871    0.8871     15997

Confusion Matrix:
[[7044  928]
 [ 878 7147]]


Epoch 5:   0%|          | 0/20495 [00:00<?, ?it/s]/tmp/ipykernel_23/401290051.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Evaluating Validation Epoch 5: 100%|██████████| 1000/1000 [05:30<00:00,  3.03it/s]



RESULTS (Validation Epoch 5)
Accuracy : 88.2853%
ROC-AUC : 0.9562
PR-AUC : 0.9565
FN Count : 975
FP Count : 899
              precision    recall  f1-score   support

        Safe     0.8789    0.8872    0.8830      7972
        Vuln     0.8869    0.8785    0.8827      8025

    accuracy                         0.8829     15997
   macro avg     0.8829    0.8829    0.8829     15997
weighted avg     0.8829    0.8829    0.8829     15997

Confusion Matrix:
[[7073  899]
 [ 975 7050]]


Evaluating Final Test: 100%|██████████| 1250/1250 [06:53<00:00,  3.02it/s]


RESULTS (Final Test)
Accuracy : 88.3727%
ROC-AUC : 0.9602
PR-AUC : 0.9612
FN Count : 1125
FP Count : 1200
              precision    recall  f1-score   support

        Safe     0.8861    0.8794    0.8828      9953
        Vuln     0.8814    0.8880    0.8847     10043

    accuracy                         0.8837     19996
   macro avg     0.8838    0.8837    0.8837     19996
weighted avg     0.8837    0.8837    0.8837     19996

Confusion Matrix:
[[8753 1200]
 [1125 8918]]
